# Volume 13, Chapter 6: The circle method I, partitions (Python companion)

Exercises first, solutions at the end. We compute $p(n)$ exactly with big integers, compare it with the Hardy–Ramanujan formula and with Rademacher's convergent series, check Ramanujan's congruences, and look at the generating function on the circle. The computer science application: **exact counting and uniform random generation of partitions**, comparing a dynamic program with Euler's recurrence and sampling partitions with a Boltzmann sampler. Requires `numpy` and `mpmath`.

In [ ]:
import math, time, random
import numpy as np
from mpmath import mp, mpf, sqrt, exp, sinh, cos, pi, diff, log

def partitions_pentagonal(N):
    # p(0..N) by Euler's pentagonal recurrence: O(N^1.5) big-integer additions
    p = [1] + [0] * N
    for n in range(1, N + 1):
        s, k = 0, 1
        while True:
            g1 = k * (3 * k - 1) // 2
            if g1 > n: break
            g2 = g1 + k
            t = p[n - g1] + (p[n - g2] if g2 <= n else 0)
            s = s + t if k % 2 else s - t
            k += 1
        p[n] = s
    return p

def hardy_ramanujan(n):
    return exp(pi * sqrt(mpf(2) * n / 3)) / (4 * sqrt(3) * n)

## Exercises

**P6.1 (★) Hardy–Ramanujan.** Compute $p(n)$ for $n\le5000$. Print $p(200)$ and the number of digits of $p(5000)$. Compute the ratio $p(n)/\mathrm{HR}(n)$ and check that $\sqrt n\,(1-\text{ratio})$ tends to $\frac1C+\frac C{48}$, $C=\pi\sqrt{2/3}$ (Figure 6.1). Check the upper bound of Exercise 6.2.

**P6.2 (★★) Rademacher's series.** Implement Theorem 6.3, with the Dedekind sums, and compute the error of the partial sums for $p(200)$ and $p(1000)$ with $1,2,\dots,10$ terms. How many terms give $p(1000)$ to the nearest integer?

**P6.3 (★) Ramanujan's congruences.** Check that $p(5m+4)\equiv0\pmod5$, $p(7m+5)\equiv0\pmod7$ and $p(11m+6)\equiv0\pmod{11}$ for all $n\le5000$. For each prime $\ell\le31$ and each residue $r$, test whether $p(\ell m+r)\equiv0\pmod\ell$ for all $m$ in range. Which pairs $(\ell,r)$ survive?

**P6.4 (★★) The generating function on the circle.** For $r=0.99$, compute $\log|P(re(x))|$ on a grid of $x$ (use $\log P(z)=\sum_m\frac1m\frac{z^m}{1-z^m}$) and the heights of the peaks at $x=\frac1k$, $k\le6$, relative to $\log P(r)$. Repeat with $r=0.999$: do the heights move towards $\frac1{k^2}$? Check Exercise 6.3 numerically: $\log P(-r)$ against $\frac{\pi^2}{24(1-r)}$.

**P6.5 (★★) Counting and sampling partitions (computer science).** (a) Compare two exact algorithms for $p(0),\dots,p(N)$: the dynamic program of the coin problem (add parts $1,2,\dots,N$ one at a time, $O(N^2)$ additions) and Euler's recurrence ($O(N^{3/2})$). Time both for $N=1000,2000,4000$. (b) A *Boltzmann sampler* draws the multiplicities $m_k$ of the parts $k=1,2,\dots$ independently, geometric with $P(m_k=j)=(1-x^k)x^{kj}$. Show that the resulting partition, conditioned on having size $n$, is uniform among the $p(n)$ partitions of $n$. With $x=e^{-\pi/\sqrt{6n}}$, sample uniform partitions of $n=1000$ by rejection, estimate the acceptance rate, and compare the average number of ones with the exact value $\sum_kp(n-k)/p(n)$ and with $\sqrt{6n}/\pi$ (Exercise 6.4).

## Solutions

### P6.1
Python integers are exact, so the recurrence gives every digit.

In [ ]:
mp.dps = 40
N = 5000
t0 = time.time(); p = partitions_pentagonal(N); t1 = time.time()
print(f"p(0..{N}) in {t1 - t0:.2f} s;  p(200) = {p[200]};  p(5000) has {len(str(p[5000]))} digits")
C = pi * sqrt(mpf(2) / 3)
print("predicted limit of sqrt(n)(1 - ratio):", float(1 / C + C / 48))
for n in [10, 100, 1000, 2000, 5000]:
    ratio = mpf(p[n]) / hardy_ramanujan(n)
    print(f"n = {n:5d}: ratio = {float(ratio):.6f},  sqrt(n)(1-ratio) = {float(sqrt(n) * (1 - ratio)):.4f}")
for n in [10, 100, 1000, 5000]:
    bound = 2 * (24 * mpf(n)) ** (-0.25) * exp(C * sqrt(n))
    print(f"n = {n:5d}: bound of Exercise 6.2 / p(n) = {float(bound / p[n]):9.1f},  n^(3/4) = {n ** 0.75:7.1f}")

**What it shows.** $p(200)=3\,972\,999\,029\,388$, the last entry of MacMahon's table, and $p(5000)$ has $75$ digits. The ratio climbs to $1$ like $1-\frac{0.443}{\sqrt n}$: the product $\sqrt n(1-\text{ratio})$ is $0.401$ at $n=10$, $0.437$ at $n=100$ and $0.442$ at $n=5000$, approaching $\frac1C+\frac C{48}=0.4433$. The error is of order $n^{-1/2}$, better than the $n^{-1/4}$ proved in Theorem 6.2. The bound of Exercise 6.2 is off by a factor that grows like $n^{3/4}$, as the solution predicts (the ratio of the two stays between $6$ and $7$ from $n=100$ on).

### P6.2
$A_k(n)=\sum_h e^{\pi i s(h,k)-2\pi inh/k}$ is real, since the terms for $h$ and $k-h$ are conjugate, so we sum the cosines. `mpmath.diff` takes the derivative numerically, to the working precision.

In [ ]:
from math import gcd
mp.dps = 60

def dedekind_sum(h, k):
    return sum(mpf(r) / k * (mpf(h * r % k) / k - mpf(1) / 2) for r in range(1, k) if (h * r) % k)

def A(k, n):
    return sum(cos(pi * dedekind_sum(h, k) - 2 * pi * n * h / mpf(k)) for h in range(k) if gcd(h, k) == 1)

def rademacher(n, K):
    total = mpf(0)
    for k in range(1, K + 1):
        f = lambda x, k=k: sinh(pi / k * sqrt(mpf(2) / 3 * (x - mpf(1) / 24))) / sqrt(x - mpf(1) / 24)
        total += A(k, n) * sqrt(k) * diff(f, n)
    return total / (pi * sqrt(2))

for n in [200, 1000]:
    print(f"n = {n}")
    for K in range(1, 11):
        err = rademacher(n, K) - p[n]
        print(f"   {K:2d} terms: error = {float(err): .6g}")

**What it shows.** For $n=200$ the errors are about $-36202$, $81$, $-6.7$, $-1.6$, $-0.14$, and after eight terms $-0.024$: once the error is below $\frac12$, rounding gives $p(200)$ exactly. For $n=1000$, whose $p(n)$ has $32$ digits, the first term alone is already within a relative error of about $2\cdot10^{-18}$, and eight terms bring the error down to $0.05$, so they give $p(1000)$ exactly after rounding; the next terms are smaller than $\frac12$ and only make the partial sums wobble. In general $O(\sqrt n)$ terms suffice. Since $p(n)\approx e^{2.565\sqrt n}$, the term $k$ is of size $e^{2.565\sqrt n/k}$, so each term is much smaller than the previous one, which is why the series converges so fast at the beginning.

### P6.3

In [ ]:
for ell, r in [(5, 4), (7, 5), (11, 6)]:
    ok = all(p[n] % ell == 0 for n in range(r, N + 1, ell))
    print(f"p({ell}m+{r}) = 0 mod {ell} for all {ell}m+{r} <= {N}: {ok}")
survivors = [(ell, r) for ell in [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31] for r in range(ell)
             if all(p[n] % ell == 0 for n in range(r, N + 1, ell))]
print("surviving (ell, r):", survivors)

**What it shows.** The three congruences hold. Among all primes $\ell\le31$ and all residues, only $(5,4)$, $(7,5)$ and $(11,6)$ survive. In 2003 Ahlgren and Boylan proved that these are the only congruences of the form $p(\ell m+r)\equiv0\pmod\ell$ with $\ell$ prime; in each case $24r\equiv1\pmod\ell$, the same $\frac1{24}$ as in the exponent $n-\frac1{24}$. (S. Ahlgren and M. Boylan, "Arithmetic properties of the partition function", *Invent. Math.* 153 (2003), 487–502.)

### P6.4

In [ ]:
def logP(z, M=None):
    # log P(z) = sum_m (1/m) z^m / (1 - z^m), truncated where |z|^m is negligible
    r = abs(z)
    M = M or int(60 / (1 - r))
    m = np.arange(1, M + 1)
    zm = z ** m
    return np.sum(zm / (1 - zm) / m)

for r in [0.99, 0.999]:
    l0 = logP(r).real
    heights = []
    for k in range(2, 7):
        xs = 1 / k + np.linspace(-0.3, 0.3, 601) * (1 - r)
        heights.append(max(logP(r * np.exp(2j * np.pi * x)).real for x in xs) / l0)
    print(f"r = {r}: log P(r) = {l0:.2f}; peak heights at 1/k, k=2..6:", " ".join(f"{h:.4f}" for h in heights))
print("1/k^2:", " ".join(f"{1 / k**2:.4f}" for k in range(2, 7)))
for r in [0.99, 0.999]:
    print(f"r = {r}: log P(-r) = {logP(-r).real:.3f}, pi^2/(24(1-r)) = {math.pi**2 / (24 * (1 - r)):.3f}")

**What it shows.** At $r=0.99$ the peak heights are $0.237$, $0.096$, $0.048$, $0.026$, $0.015$ against $\frac1{k^2}=0.25$, $0.111$, $0.0625$, $0.04$, $0.028$; at $r=0.999$ they are much closer. The approach is slow because the next term, $\frac12\log(1-r)$ in Lemma 6.1, has a different size at each cusp. The value at $-r$ matches $\frac{\pi^2}{24(1-r)}$ up to a lower-order term, as Exercise 6.3 proves.

### P6.5
(b) The probability of a given partition with multiplicities $(m_k)$ is $\prod_k(1-x^k)x^{km_k}=\frac{x^{\sum km_k}}{P(x)}=\frac{x^n}{P(x)}$: it depends only on the size $n$. So, conditioned on the size, all partitions of $n$ are equally likely. The choice $x=e^{-\pi/\sqrt{6n}}$ makes the expected size close to $n$: it is the saddle point of Exercise 6.2.

In [ ]:
def partitions_dp(N):
    p = [1] + [0] * N
    for part in range(1, N + 1):
        for n in range(part, N + 1):
            p[n] += p[n - part]
    return p

for M in [1000, 2000, 4000]:
    t0 = time.time(); a = partitions_dp(M); t1 = time.time(); b = partitions_pentagonal(M); t2 = time.time()
    assert a == b
    print(f"N = {M}: dynamic program {t1 - t0:6.2f} s,  pentagonal {t2 - t1:6.3f} s")

def boltzmann_partition(n, rng):
    x = math.exp(-math.pi / math.sqrt(6 * n))
    k = np.arange(1, n + 1)
    q = x ** k
    attempts = 0
    while True:
        attempts += 1
        # number of copies of k is geometric: P(m = j) = (1 - q) q^j
        m = rng.geometric(1 - q) - 1
        if (k * m).sum() == n:
            return m, attempts

rng = np.random.default_rng(1)
n = 1000
t0 = time.time()
samples, tries = [], 0
while len(samples) < 2000:
    m, a = boltzmann_partition(n, rng)
    samples.append(m); tries += a
print(f"2000 uniform partitions of {n} in {time.time() - t0:.1f} s, acceptance rate {len(samples) / tries:.4f}")
ones = np.array([m[0] for m in samples])
exact = sum(p[n - k] for k in range(1, n + 1)) / p[n]
print(f"average number of ones: sampled {ones.mean():.2f} +- {ones.std() / math.sqrt(len(ones)):.2f}, exact {exact:.2f}, sqrt(6n)/pi = {math.sqrt(6 * n) / math.pi:.2f}")

**What it shows.** (a) The dynamic program does about $\frac{N^2}2$ big-integer additions, the pentagonal recurrence about $\frac43N^{3/2}$, so doubling $N$ multiplies the first time by about $4$ and the second by about $3$ (more for both, as the integers grow). At $N=4000$ the pentagonal recurrence is about $17$ times faster. (b) The sampler is exactly uniform; each attempt costs $O(n)$ operations, and the acceptance probability is of order $n^{-3/4}$, here the measured acceptance rate is $0.0017$, so a uniform partition of $1000$ costs about $600$ attempts. The average number of ones is $24.6\pm0.6$, matching the exact value $24.47$; the asymptotic value $\frac{\sqrt{6n}}\pi=24.66$ is slightly larger, as Exercise 6.4 allows. Under the Boltzmann model, $m_1$ is geometric with mean $\frac x{1-x}\approx\frac{\sqrt{6n}}\pi$, which is the probabilistic reason for Exercise 6.4.

*References.* B. Fristedt, "The structure of random partitions of large integers", *Trans. Amer. Math. Soc.* 337 (1993), 703–735. P. Duchon, P. Flajolet, G. Louchard and G. Schaeffer, "Boltzmann samplers for the random generation of combinatorial structures", *Combin. Probab. Comput.* 13 (2004), 577–625.